In [1]:
import os
os.environ["HF_HOME"] = "/kaggle/temp/hf"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("fitcheck")   # was: ["fitcheck"]
    print("HF token loaded")
except Exception as e:
    print("No HF token:", e)

HF token loaded


In [2]:
import os
if not os.path.isdir("/kaggle/working/fitcheck/.git"):
    !git clone https://github.com/Anassbzdd/fitcheck.git /kaggle/working/fitcheck
%cd /kaggle/working/fitcheck
!git pull --ff-only
!git log --oneline -1

Cloning into '/kaggle/working/fitcheck'...
remote: Enumerating objects: 311, done.
remote: Counting objects: 100% (311/311), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 311 (delta 181), reused 207 (delta 81), pack-reused 0 (from 0)
Receiving objects: 100% (311/311), 454.81 KiB | 11.97 MiB/s, done.
Resolving deltas: 100% (181/181), done.
/kaggle/working/fitcheck
Already up to date.
b1e557b (HEAD -> main, origin/main, origin/HEAD) small fixes


In [3]:
!pip install -q -U transformers peft accelerate bitsandbytes
!pip install -q -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 91.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.9 MB/s eta 0:00:00:00:01
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fitcheck-llm (pyproject.toml) ... done


In [4]:
!python -c "import torch; print(torch.__version__, torch.cuda.get_device_name(0), \
    f'{torch.cuda.get_device_properties(0).total_memory/1024**2:,.0f} MiB visible to torch')"
!python scripts/measure.py --help | head -5

2.10.0+cu128 Tesla T4 14,912 MiB visible to torch
usage: measure.py [-h] [--quant {none,nf4,int8}] [--double-quant] [--qlora]
                  [--precision {fp32,fp16,bf16}] [--lora-r LORA_R] [--no-lora]
                  [--lora-targets LORA_TARGETS] [--batch-size BATCH_SIZE]
                  [--seq-len SEQ_LEN]
                  [--optimizer {adamw,adam8bit,sgd,sgd-momentum}]


In [5]:
!du -sh /kaggle/temp/hf 2>/dev/null; df -h /kaggle/temp | tail -1

df: /kaggle/temp: No such file or directory


In [6]:
!python scripts/measure.py TinyLlama/TinyLlama-1.1B-Chat-v1.0 --qlora --precision fp16 --lora-r 32 --batch-size 2 --seq-len 512  --gpu t4


config.json: 100%|█████████████████████████████| 608/608 [00:00<00:00, 3.30MB/s]
measure.py: 2 GPUs visible; measuring device 0 (Tesla T4) only. This is by design -- fitcheck predicts single-device memory, so a sharded or DDP run would not be comparable to the prediction.
model.safetensors: 100%|████████████████████| 2.20G/2.20G [00:08<00:00, 267MB/s]
generation_config.json: 100%|███████████████████| 124/124 [00:00<00:00, 651kB/s]

  TinyLlama/TinyLlama-1.1B-Chat-v1.0  on  Tesla T4
  QLoRA r=32 [q,k,v,o], bs=2, seq=512, fp16, adamw, ckpt, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 1,109,059,584 logical | 9,011,200 trainable
          base 1,100,048,384 vs fitcheck P 1,100,048,384  (+0)  [OK]
  optimizer states: 69 MiB observed, dtype float32
  step time: 0.80s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105

In [7]:
!python scripts/measure.py TinyLlama/TinyLlama-1.1B-Chat-v1.0 --qlora --precision fp16 --lora-r 32 --batch-size 2 --seq-len 1024 --gpu t4


measure.py: 2 GPUs visible; measuring device 0 (Tesla T4) only. This is by design -- fitcheck predicts single-device memory, so a sharded or DDP run would not be comparable to the prediction.
Loading weights: 100%|███████████████████████| 201/201 [00:01<00:00, 106.63it/s]

  TinyLlama/TinyLlama-1.1B-Chat-v1.0  on  Tesla T4
  QLoRA r=32 [q,k,v,o], bs=2, seq=1024, fp16, adamw, ckpt, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 1,109,059,584 logical | 9,011,200 trainable
          base 1,100,048,384 vs fitcheck P 1,100,048,384  (+0)  [OK]
  optimizer states: 69 MiB observed, dtype float32
  step time: 2.00s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                    1,074 MiB
    gradients (after backward)                 34 MiB
    peak allocated  (tensor bytes)          2,87

In [8]:
!python scripts/measure.py TinyLlama/TinyLlama-1.1B-Chat-v1.0 --qlora --precision fp16 --lora-r 32 --batch-size 2 --seq-len 2048 --gpu t4


measure.py: 2 GPUs visible; measuring device 0 (Tesla T4) only. This is by design -- fitcheck predicts single-device memory, so a sharded or DDP run would not be comparable to the prediction.
Loading weights: 100%|███████████████████████| 201/201 [00:01<00:00, 105.23it/s]

  TinyLlama/TinyLlama-1.1B-Chat-v1.0  on  Tesla T4
  QLoRA r=32 [q,k,v,o], bs=2, seq=2048, fp16, adamw, ckpt, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 1,109,059,584 logical | 9,011,200 trainable
          base 1,100,048,384 vs fitcheck P 1,100,048,384  (+0)  [OK]
  optimizer states: 69 MiB observed, dtype float32
  step time: 5.98s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
    allocated after load                    1,074 MiB
    gradients (after backward)                 34 MiB
    peak allocated  (tensor bytes)          6,65

In [9]:
!python scripts/measure.py TinyLlama/TinyLlama-1.1B-Chat-v1.0 --qlora --precision fp16 --lora-r 32 --batch-size 2 --seq-len 512  --gpu t4 --flash-attn --attn-impl sdpa


measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
measure.py: 2 GPUs visible; measuring device 0 (Tesla T4) only. This is by design -- fitcheck predicts single-device memory, so a sharded or DDP run would not be comparable to the prediction.
Loading weights: 100%|███████████████████████| 201/201 [00:01<00:00, 106.29it/s]

  TinyLlama/TinyLlama-1.1B-Chat-v1.0  on  Tesla T4
  QLoRA r=32 [q,k,v,o], bs=2, seq=512, fp16, adamw, ckpt, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 1,109,059,584 logical | 9,011,200 trainable
          base 1,100,048,384 vs fitcheck P 1,100,048,384  (+0)  [OK]
  optimizer states: 69 MiB observed, dtype float32
  step time: 0.72s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for

In [10]:
!python scripts/measure.py TinyLlama/TinyLlama-1.1B-Chat-v1.0 --qlora --precision fp16 --lora-r 32 --batch-size 2 --seq-len 1024 --gpu t4 --flash-attn --attn-impl sdpa


measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
measure.py: 2 GPUs visible; measuring device 0 (Tesla T4) only. This is by design -- fitcheck predicts single-device memory, so a sharded or DDP run would not be comparable to the prediction.
Loading weights: 100%|███████████████████████| 201/201 [00:01<00:00, 107.57it/s]

  TinyLlama/TinyLlama-1.1B-Chat-v1.0  on  Tesla T4
  QLoRA r=32 [q,k,v,o], bs=2, seq=1024, fp16, adamw, ckpt, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 1,109,059,584 logical | 9,011,200 trainable
          base 1,100,048,384 vs fitcheck P 1,100,048,384  (+0)  [OK]
  optimizer states: 69 MiB observed, dtype float32
  step time: 1.59s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, fo

In [11]:
!python scripts/measure.py TinyLlama/TinyLlama-1.1B-Chat-v1.0 --qlora --precision fp16 --lora-r 32 --batch-size 2 --seq-len 2048 --gpu t4 --flash-attn --attn-impl sdpa


measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
measure.py: 2 GPUs visible; measuring device 0 (Tesla T4) only. This is by design -- fitcheck predicts single-device memory, so a sharded or DDP run would not be comparable to the prediction.
Loading weights: 100%|███████████████████████| 201/201 [00:01<00:00, 107.00it/s]

  TinyLlama/TinyLlama-1.1B-Chat-v1.0  on  Tesla T4
  QLoRA r=32 [q,k,v,o], bs=2, seq=2048, fp16, adamw, ckpt, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 1,109,059,584 logical | 9,011,200 trainable
          base 1,100,048,384 vs fitcheck P 1,100,048,384  (+0)  [OK]
  optimizer states: 69 MiB observed, dtype float32
  step time: 3.95s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, fo

In [12]:
!python scripts/measure.py HuggingFaceTB/SmolLM2-1.7B --qlora --precision fp16 --lora-r 32 --batch-size 4 --seq-len 1024 --gpu t4


config.json: 100%|█████████████████████████████| 635/635 [00:00<00:00, 4.23MB/s]
measure.py: 2 GPUs visible; measuring device 0 (Tesla T4) only. This is by design -- fitcheck predicts single-device memory, so a sharded or DDP run would not be comparable to the prediction.
model.safetensors: 100%|████████████████████| 3.42G/3.42G [00:11<00:00, 294MB/s]
generation_config.json: 100%|███████████████████| 111/111 [00:00<00:00, 578kB/s]

  HuggingFaceTB/SmolLM2-1.7B  on  Tesla T4
  QLoRA r=32 [q,k,v,o], bs=4, seq=1024, fp16, adamw, ckpt, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 1,723,959,296 logical | 12,582,912 trainable
          base 1,711,376,384 vs fitcheck P 1,711,376,384  (+0)  [OK]
  optimizer states: 96 MiB observed, dtype float32
  step time: 6.39s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 MiB
 

In [13]:
!python scripts/measure.py HuggingFaceTB/SmolLM2-1.7B --qlora --precision fp16 --lora-r 32 --batch-size 4 --seq-len 1024 --gpu t4 --flash-attn --attn-impl sdpa


measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
measure.py: 2 GPUs visible; measuring device 0 (Tesla T4) only. This is by design -- fitcheck predicts single-device memory, so a sharded or DDP run would not be comparable to the prediction.
Loading weights: 100%|████████████████████████| 218/218 [00:03<00:00, 68.39it/s]

  HuggingFaceTB/SmolLM2-1.7B  on  Tesla T4
  QLoRA r=32 [q,k,v,o], bs=4, seq=1024, fp16, adamw, ckpt, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 1,723,959,296 logical | 12,582,912 trainable
          base 1,711,376,384 vs fitcheck P 1,711,376,384  (+0)  [OK]
  optimizer states: 96 MiB observed, dtype float32
  step time: 4.77s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref) 

In [14]:
!python scripts/measure.py Qwen/Qwen2.5-1.5B-Instruct --qlora --precision fp16 --lora-r 32 --batch-size 2 --seq-len 1024 --gpu t4


config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 4.46MB/s]
measure.py: 2 GPUs visible; measuring device 0 (Tesla T4) only. This is by design -- fitcheck predicts single-device memory, so a sharded or DDP run would not be comparable to the prediction.
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:12<00:00, 241MB/s]
generation_config.json: 100%|██████████████████| 242/242 [00:00<00:00, 1.77MB/s]

  Qwen/Qwen2.5-1.5B-Instruct  on  Tesla T4
  QLoRA r=32 [q,k,v,o], bs=2, seq=1024, fp16, adamw, ckpt, no FA

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: eager   warmup: 1   measured steps: 1
  sdpa backend: n/a (not sdpa)
  params: 1,552,430,592 logical | 8,716,288 trainable
          base 1,543,714,304 vs fitcheck P 1,543,656,960  (+57,344)  [OK]
  optimizer states: 66 MiB observed, dtype float32
  step time: 2.62s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for ref)           105 M

In [15]:
!python scripts/measure.py Qwen/Qwen2.5-1.5B-Instruct --qlora --precision fp16 --lora-r 32 --batch-size 2 --seq-len 1024 --gpu t4 --flash-attn --attn-impl sdpa


measure.py: Tesla T4 is sm_75, so Flash Attention 2 is unavailable. --flash-attn is being applied to the PREDICTION only; the measured kernel is 'sdpa'.
measure.py: 2 GPUs visible; measuring device 0 (Tesla T4) only. This is by design -- fitcheck predicts single-device memory, so a sharded or DDP run would not be comparable to the prediction.
Loading weights: 100%|███████████████████████| 338/338 [00:02<00:00, 125.67it/s]

  Qwen/Qwen2.5-1.5B-Instruct  on  Tesla T4
  QLoRA r=32 [q,k,v,o], bs=2, seq=1024, fp16, adamw, ckpt, attn=sdpa

  torch 2.10.0+cu128 | CUDA 12.8 | python 3.12.13 | Linux
  attention: sdpa   warmup: 1   measured steps: 1
  sdpa backend: EFFICIENT_ATTENTION (+ GQA expand shim)
  params: 1,552,430,592 logical | 8,716,288 trainable
          base 1,543,714,304 vs fitcheck P 1,543,656,960  (+57,344)  [OK]
  optimizer states: 66 MiB observed, dtype float32
  step time: 2.48s

  MEASURED
    CUDA context (at peak)                    141 MiB
    CUDA context (at init, for r